# Амортизация, шаг 1: справочник цен в озере + проверка моделей

1. Читает `term_model.xlsx` (`model_name`, `price`), нормализует модель так же, как `01_07_build_amortization_drp.ipynb`, и ставит флаг исключения (Tactilion, VeriFone, Ingenico, IRAS, Pax S80/S90/D210).
2. Загружает справочник в `sandbox_ai.shestopalov_terminal_model_price` (новая таблица, перезаписывается целиком только она).
3. Проверяет `dwh.bmrt_bm_det_device`: колонки, свежесть, совпадение моделей и цен со старой таблицей `..._jan_aug` за август 2026.

Больше ничего не создаёт и не меняет. Логин для keytab спрашивается один раз.

In [ ]:
import re

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

EXCEL_PATH = "/home/jovyan/documents/Equaring/Data/term_model.xlsx"
SOURCE_FILE = "term_model.xlsx"
PRICE_TABLE = "sandbox_ai.shestopalov_terminal_model_price"
DEVICE_TABLE = "dwh.bmrt_bm_det_device"
OLD_AMORT_TABLE = "sandbox_ai.shestopalov_terminal_amortization_model_jan_aug"
CHECK_MONTH_START = "2026-08-01"


def norm_model(v):
    if v is None or pd.isna(v):
        return None
    s = str(v).strip().lower().replace("\xa0", " ")
    if s in {"none", "nan", "null", "<null>", "nat"}:
        return None
    s = s.replace("ё", "е")
    s = re.sub(r"\s+", " ", s)
    return s or None


EXCLUDE_BRAND_RE = re.compile(r"(tactilion|verifone|veri\s*fone|ingenico|iras)", re.IGNORECASE)


def is_excluded_model(model):
    if not model:
        return 0
    t = str(model).lower()
    if EXCLUDE_BRAND_RE.search(t):
        return 1
    if "pax" in t and any(tok in t.replace(" ", "") for tok in ("s80", "s90", "d210")):
        return 1
    return 0


if "imp" not in globals():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={"user_name": input("Login (keytab): ").strip()},
    )


def imp_fetch(sql):
    with imp:
        imp.execute("SET MEM_LIMIT=4g")
        return imp.fetch(sql)


def imp_exec(sql):
    with imp:
        imp.execute(sql)

## 1. Цены из Excel

Если у одной модели после нормализации разные цены, загрузка остановится: сначала поправьте Excel.

In [ ]:
raw = pd.read_excel(EXCEL_PATH)
missing_cols = [c for c in ("model_name", "price") if c not in raw.columns]
if missing_cols:
    raise RuntimeError(f"В {SOURCE_FILE} нет колонок {missing_cols}. Есть: {raw.columns.tolist()}")

prices = raw[["model_name", "price"]].copy()
prices["model_key"] = prices["model_name"].map(norm_model)
prices["price"] = pd.to_numeric(prices["price"], errors="coerce")

bad_rows = prices[prices["model_key"].isna() | prices["price"].isna()]
print("Строк в Excel:", len(raw), "| без модели или цены (пропускаются):", len(bad_rows))
if len(bad_rows):
    display(bad_rows)
prices = prices.dropna(subset=["model_key", "price"])

conflicts = prices.groupby("model_key")["price"].nunique()
conflicts = prices[prices["model_key"].isin(conflicts[conflicts > 1].index)].sort_values("model_key")
if len(conflicts):
    display(conflicts)
    raise RuntimeError("У одной модели разные цены (таблица выше). Поправьте Excel и запустите снова.")

price_ref = (
    prices.groupby("model_key", as_index=False)
    .agg(model_name=("model_name", "first"), price=("price", "max"))
)
price_ref["is_excluded"] = price_ref["model_key"].map(is_excluded_model)
price_ref["amortization_monthly"] = (price_ref["price"] / 48.0).where(price_ref["is_excluded"] == 0, 0.0)

print("Моделей в справочнике:", len(price_ref), "| из них исключены:", int(price_ref["is_excluded"].sum()))
display(price_ref.sort_values(["is_excluded", "model_key"]))

## 2. Загрузка справочника в озеро

Перезаписывается только `sandbox_ai.shestopalov_terminal_model_price`.

In [ ]:
def sql_str(v):
    return "'" + str(v).replace("\\", "\\\\").replace("'", "\\'") + "'"


load_dt = pd.Timestamp.now().strftime("%Y-%m-%d")
values_sql = ",\n".join(
    f"({sql_str(r.model_key)}, {sql_str(r.model_name)}, cast({float(r.price)} as double), "
    f"cast({int(r.is_excluded)} as int), cast({float(r.amortization_monthly)} as double), "
    f"{sql_str(SOURCE_FILE)}, {sql_str(load_dt)})"
    for r in price_ref.itertuples()
)

imp_exec(f"""
    CREATE EXTERNAL TABLE IF NOT EXISTS {PRICE_TABLE} (
        model_key string,
        model_name string,
        price double,
        is_excluded int,
        amortization_monthly double,
        source_file string,
        load_dt string
    )
    STORED AS PARQUET
    TBLPROPERTIES ('external.table.purge'='true')
""")
imp_exec(f"INSERT OVERWRITE TABLE {PRICE_TABLE} VALUES\n{values_sql}")
imp_exec(f"REFRESH {PRICE_TABLE}")

check = imp_fetch(f"""
    SELECT count(*) AS rows_cnt, count(DISTINCT model_key) AS models, sum(is_excluded) AS excluded,
           max(load_dt) AS load_dt
    FROM {PRICE_TABLE}
""")
display(check)
if int(check.rows_cnt.iloc[0]) != len(price_ref) or int(check.models.iloc[0]) != len(price_ref):
    raise RuntimeError("В озере не столько моделей, сколько в Excel")
print("OK: справочник загружен")

## 3. Таблица моделей в озере `dwh.bmrt_bm_det_device`

Колонки, число строк и свежесть (максимум по колонкам с датой).

In [ ]:
imp_exec(f"REFRESH {DEVICE_TABLE}")
desc = imp_fetch(f"DESCRIBE {DEVICE_TABLE}")
desc.columns = [str(c).lower() for c in desc.columns]
device_cols = desc.iloc[:, 0].str.lower().tolist()
print("Колонки:", device_cols)

need = [c for c in ("id_device", "code_device", "model_device") if c not in device_cols]
print("Нет нужных колонок:", need or "все на месте")

date_cols = [c for c in device_cols if any(t in c for t in ("date", "_dt", "dt_", "load", "update"))][:5]
agg = ", ".join(["count(*) AS rows_cnt"] + [f"max({c}) AS max_{c}" for c in date_cols])
display(imp_fetch(f"SELECT {agg} FROM {DEVICE_TABLE}"))
display(imp_fetch(f"SELECT * FROM {DEVICE_TABLE} LIMIT 5"))

## 4. Сверка с разовой таблицей за август 2026

Для терминалов из `..._jan_aug` за август берём модель из озера (сначала по `id_device`, потом по `code_device`) и сравниваем:
- нашлась ли модель;
- совпадает ли она с моделью из ЦХД в старой таблице;
- нашлась ли цена в новом справочнике и совпадает ли месячная амортизация.

In [ ]:
cmp = imp_fetch(f"""
    WITH old AS (
        SELECT cast(c_nter AS string) AS c_nter,
               model_device AS old_model,
               cast(amortization_monthly AS double) AS old_amort_monthly,
               cast(amortization_for_report_month AS double) AS old_amort_month
        FROM {OLD_AMORT_TABLE}
        WHERE cast(snapshot_month_start AS string) = '{CHECK_MONTH_START}'
    ),
    dev AS (
        SELECT trim(cast(id_device AS string)) AS id_device,
               trim(cast(code_device AS string)) AS code_device,
               trim(cast(model_device AS string)) AS model_device
        FROM {DEVICE_TABLE}
        WHERE model_device IS NOT NULL AND trim(cast(model_device AS string)) <> ''
    ),
    by_id AS (
        SELECT id_device, max(model_device) AS model_device FROM dev GROUP BY id_device
    ),
    by_code AS (
        SELECT code_device, max(model_device) AS model_device FROM dev GROUP BY code_device
    )
    SELECT o.c_nter, o.old_model, o.old_amort_monthly, o.old_amort_month,
           coalesce(i.model_device, c.model_device) AS lake_model,
           CASE WHEN i.model_device IS NOT NULL THEN 'id_device'
                WHEN c.model_device IS NOT NULL THEN 'code_device' END AS join_key
    FROM old o
    LEFT JOIN by_id i
      ON i.id_device = CASE WHEN o.c_nter RLIKE '^[0-9]+$'
                            THEN cast(cast(o.c_nter AS bigint) AS string) ELSE o.c_nter END
    LEFT JOIN by_code c
      ON upper(c.code_device) = upper(o.c_nter)
""")

cmp["old_key"] = cmp["old_model"].map(norm_model)
cmp["lake_key"] = cmp["lake_model"].map(norm_model)
cmp = cmp.merge(price_ref[["model_key", "amortization_monthly"]].rename(
    columns={"model_key": "lake_key", "amortization_monthly": "new_amort_monthly"}), on="lake_key", how="left")

n = len(cmp)
found = cmp["lake_key"].notna()
same_model = found & (cmp["lake_key"] == cmp["old_key"])
old_m = pd.to_numeric(cmp["old_amort_monthly"], errors="coerce").fillna(0)
new_m = pd.to_numeric(cmp["new_amort_monthly"], errors="coerce").fillna(0)
same_amort = (old_m - new_m).abs() < 0.01

summary = pd.DataFrame([
    {"проверка": "терминалов в августе", "значение": n},
    {"проверка": "модель нашлась в озере, %", "значение": round(100 * found.mean(), 2)},
    {"проверка": "модель совпала с ЦХД, %", "значение": round(100 * same_model.sum() / max(found.sum(), 1), 2)},
    {"проверка": "месячная амортизация совпала, %", "значение": round(100 * same_amort.mean(), 2)},
    {"проверка": "сумма месячной амортизации: старая", "значение": round(old_m.sum(), 2)},
    {"проверка": "сумма месячной амортизации: новая", "значение": round(new_m.sum(), 2)},
])
display(summary)

print("Чем найдены модели:")
display(cmp["join_key"].fillna("не нашлась").value_counts().to_frame("terminals"))

print("Топ-20 расхождений амортизации (по модели):")
diff = cmp[~same_amort].assign(old_m=old_m, new_m=new_m)
display(
    diff.groupby(["old_model", "lake_model"], dropna=False, as_index=False)
    .agg(terminals=("c_nter", "size"), old_m=("old_m", "max"), new_m=("new_m", "max"))
    .sort_values("terminals", ascending=False)
    .head(20)
)

print("Топ-20 моделей из озера без цены в Excel:")
display(
    cmp[found & cmp["new_amort_monthly"].isna()]
    .groupby("lake_model", as_index=False).size()
    .sort_values("size", ascending=False).head(20)
)